# Grands modèles multimodaux (Eden AI) face à gemma4

**Question** : des modèles nettement plus gros que gemma4 (Gemini Pro, GPT phare, Claude Opus, grand Qwen3-VL…)
lisent-ils mieux les noms et prénoms manuscrits ? Et à quel coût ?

| Étape | Ce qu'elle fait | Coût Eden AI |
|---|---|---|
| 1 | Réglages et outils | — |
| 2 | Bulletins, références, images des blocs | — |
| 3 | Clés (service SSP Cloud, Eden AI) | — |
| 4 | Fonction de lecture commune (parallèle, cache, progression) | — |
| 5 | **Essai : 3 bulletins par modèle** (format, temps, consommation) | très faible |
| 6 | gemma4 sur tous les bulletins (référence) | aucun |
| 7 | Modèles retenus après l'essai, sur tous les bulletins | **principal** |
| 8 | Comparaison : justes, avoués, erreurs silencieuses, McNemar contre gemma4 | — |
| 9 | Enregistrement MLflow (projet + personnel), tableaux agrégés seulement | — |

Toutes les lectures sont mises en cache : relancer une cellule ne refacture jamais un bulletin déjà lu.

In [1]:
# 1. Reglages et outils
import base64
import contextlib
import getpass
import io
import json
import math
import os
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path

import mlflow
import pandas as pd
import requests
import s3fs
from PIL import Image
from scipy import stats

CAMPAGNE = "RG25"
TEST_RAPIDE = False                 # True : 10 bulletins (etapes 6 et 7)
N_ESSAI = 3                         # etape 5 : bulletins par modele pour l'essai

# Modeles a essayer (etape 5) : identifiants EXACTS du catalogue Eden AI, versions figees de preference
MODELES_EDEN = [
    "google/gemini-flash-latest",   # exemple : remplacer / completer par les modeles phares
]
# Modeles retenus apres l'essai (etape 7) : a remplir a la main apres l'etape 5
MODELES_RETENUS = []

MODELE_GEMMA = "gemma4-26b-moe"
URL_SSP = "https://llm.lab.sspcloud.fr/api/chat/completions"
URL_EDEN = "https://api.edenai.run/v3/chat/completions"
N_PARALLELE = 4

BUCKET = "projet-mesure-qualite-rp"
FIQUAL = {"RG25": "RP_PMQ_FIQUAL2599.txt", "RG26": "RP_PMQ_FIQUAL2699.txt"}
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats" / "eden"
BLOCS = Path("/tmp/eden/blocs")
for d in [RESULTATS, BLOCS]:
    d.mkdir(parents=True, exist_ok=True)
SUFFIXE = "_rapide" if TEST_RAPIDE else ""

PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""


def normaliser(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - m), min(1.0, c + m)


def ic(k, n):
    lo, hi = wilson(k, n)
    return f"[{lo * 100:.1f} ; {hi * 100:.1f}]"


def mcnemar(a_ok, b_ok):
    """(lignes justes pour a seul, justes pour b seul, p exact)."""
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n10, n01, (stats.binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0)


def lire_vault(chemin):
    """Secret Vault (KV v2) par l'API HTTP (la commande vault n'existe pas sur tous les services). {} si echec."""
    adresse, jeton = os.environ.get("VAULT_ADDR"), os.environ.get("VAULT_TOKEN")
    if not adresse or not jeton:
        return {}
    montage, reste = chemin.split("/", 1)
    try:
        r = requests.get(f"{adresse.rstrip('/')}/v1/{montage}/data/{reste}",
                         headers={"X-Vault-Token": jeton}, timeout=30)
        return r.json()["data"]["data"] if r.status_code == 200 else {}
    except Exception:
        return {}


print(f"campagne {CAMPAGNE} | mode {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'} | "
      f"{len(MODELES_EDEN)} modele(s) a essayer")

/home/onyxia/work/mesure-qualite-rp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


campagne RG25 | mode COMPLET | 1 modele(s) a essayer


## 2. Bulletins, références et images des blocs

Les blocs (nom + prénom, avec leurs libellés imprimés) sont découpés une fois et gardés dans `/tmp/eden/blocs`.

In [2]:
# 2. Bulletins, references, blocs
def connexion_s3():
    for essai in range(2):
        s3fs.S3FileSystem.clear_instance_cache()
        fs_ = s3fs.S3FileSystem()
        try:
            fs_.ls(BUCKET)
            return fs_
        except PermissionError:
            if essai == 1:
                raise
            print("identifiants S3 refuses : copier les nouveaux depuis Onyxia > Mon compte > connexion au stockage")
            for v in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
                os.environ[v] = getpass.getpass(f"{v} : ").strip()


fs = connexion_s3()
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
with fs.open(f"{BUCKET}/data/raw/{CAMPAGNE}/{FIQUAL[CAMPAGNE]}", "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq["cle"] = fq[POS_UT].str.strip() + "_" + fq[POS_CAB].str.strip()
ref_df = fq[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref_df.columns = ["cle", "NOM", "PRENOM"]
APP = (bi.merge(ref_df, on="cle", how="inner").drop_duplicates("cle")
       .sort_values("cle").reset_index(drop=True))
APP = APP.head(10) if TEST_RAPIDE else APP
REF = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in COLS}


def bloc(cle):
    fb = BLOCS / f"{CAMPAGNE}_{cle}.png"
    if not fb.exists():
        chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
        with fs.open(chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H))).save(fb)
    return Image.open(fb)


manquants = [c for c in APP.cle if not (BLOCS / f"{CAMPAGNE}_{c}.png").exists()]
if manquants:
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(bloc, manquants))
print(f"{len(APP)} bulletins, {len(REF)} champs de reference, blocs prets dans {BLOCS}")

169 bulletins, 338 champs de reference, blocs prets dans /tmp/eden/blocs


## 3. Les clés

Les clés ne sont jamais affichées ni écrites dans le notebook.

In [ ]:
# 3. Cles
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = (lire_vault("onyxia-kv/farisazibert/LLM").get("OPENAI_API_KEY")
                                    or getpass.getpass("cle du service LLM du SSP Cloud : ").strip())
if not os.environ.get("EDENAI_API_KEY"):
    os.environ["EDENAI_API_KEY"] = (lire_vault("onyxia-kv/farisazibert/EDENAI").get("EDENAI_API_KEY")
                                    or getpass.getpass("cle Eden AI : ").strip())
print("cle SSP Cloud :", "presente" if os.environ.get("OPENAI_API_KEY") else "ABSENTE")
print("cle Eden AI   :", "presente" if os.environ.get("EDENAI_API_KEY") else "ABSENTE")

## 4. La lecture commune

Une seule fonction appelle gemma4 (SSP Cloud) ou un modèle Eden AI : même image, même consigne, température 0.
`lectures(...)` lit tous les bulletins en parallèle, garde chaque lecture réussie dans un cache CSV, ne met pas les
échecs en cache (ils seront retentés), et affiche sa progression toutes les 30 secondes, même sans réponse.

In [4]:
# 4. Lecture commune (gemma4 ou Eden AI)
VERSIONS = {}          # modele demande -> version reellement servie


def appel(modele, image, consigne=PROMPT_BLOC, eden=True, delai=90, tentatives=2):
    """Renvoie ({champ: texte}, reussi, infos). infos : secondes, jetons, cout (si fournis)."""
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    b64 = base64.b64encode(t.getvalue()).decode()
    url = URL_EDEN if eden else URL_SSP
    cle_api = os.environ["EDENAI_API_KEY"] if eden else os.environ["OPENAI_API_KEY"]
    entetes = {"Authorization": f"Bearer {cle_api}", "Content-Type": "application/json"}
    if eden:
        entetes["x-edenai-metadata"] = "enabled"
    corps = {"model": modele, "stream": False, "temperature": 0, "max_tokens": 300,
             "messages": [{"role": "user", "content": [
                 {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                 {"type": "text", "text": consigne}]}]}
    erreur = ""
    for essai in range(tentatives):
        t0 = time.time()
        try:
            r = requests.post(url, headers=entetes, json=corps, timeout=delai)
            r.raise_for_status()
            rep = r.json()
            texte = rep["choices"][0]["message"]["content"] or ""
            m = re.search(r"\{.*\}", texte, re.S)
            try:
                d = json.loads(m.group(0)) if m else {}
            except json.JSONDecodeError:
                d = {}
            VERSIONS[modele] = rep.get("model", modele)
            usage = rep.get("usage", {}) or {}
            infos = {"secondes": time.time() - t0, "json_ok": bool(d),
                     "jetons": usage.get("total_tokens"), "cout": rep.get("cost", usage.get("cost"))}
            return {c: str(d.get(c, "") or "") for c in COLS}, True, infos
        except Exception as e:
            reponse = getattr(e, "response", None)
            erreur = f"{type(e).__name__} {getattr(reponse, 'status_code', '')} {getattr(reponse, 'text', '')[:150]}"
            time.sleep(3 * (essai + 1))
    return {c: "" for c in COLS}, False, {"erreur": erreur}


def nom_fichier(modele):
    return re.sub(r"[^a-z0-9]+", "_", modele.lower()).strip("_")


def lectures(modele, eden=True, cles=None):
    """Lectures de tous les bulletins par un modele ; cache CSV ; renvoie cle, champ, lu."""
    cles = list(APP.cle) if cles is None else cles
    cache = RESULTATS / f"lectures_{nom_fichier(modele)}_{CAMPAGNE}{SUFFIXE}.csv"
    L_ = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
    faire = [c for c in cles if c not in set(L_.cle)]
    print(f"{modele} : {len(cles) - len(faire)} deja lus, {len(faire)} a lire", flush=True)
    rows, echecs, temps, k = [], [], [], 0
    t_debut = dernier = time.time()

    def une(cle):
        res, ok, infos = appel(modele, bloc(cle), eden=eden)
        return cle, res, ok, infos

    try:
        with ThreadPoolExecutor(N_PARALLELE) as ex:
            en_cours = {ex.submit(une, c) for c in faire}
            while en_cours:
                finis, en_cours = wait(en_cours, timeout=30, return_when=FIRST_COMPLETED)
                for f in finis:
                    k += 1
                    cle, res, ok, infos = f.result()
                    if ok:
                        rows += [{"cle": cle, "champ": c, "lu": res[c]} for c in COLS]
                        temps.append(infos["secondes"])
                    else:
                        echecs.append((cle, infos.get("erreur", "")))
                if len(rows) >= 10 * len(COLS) or not en_cours:
                    L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
                    L_.to_csv(cache, index=False)
                    rows = []
                if time.time() - dernier >= 30 or not en_cours:
                    dernier = time.time()
                    ecoule = time.time() - t_debut
                    vitesse = f"{sum(temps[-20:]) / len(temps[-20:]):.1f} s/appel" if temps else "aucune reponse encore"
                    print(f"  {modele} : {k}/{len(faire)} | {ecoule / 60:.1f} min | {vitesse} | echecs {len(echecs)}",
                          flush=True)
    finally:
        if rows:
            L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
            L_.to_csv(cache, index=False)
    if echecs:
        print(f"  {len(echecs)} echecs (relancer pour les retenter). Premier : {echecs[0][1][:200]}")
    return L_[L_.cle.isin(set(cles))]


print("outils de lecture prets")

outils de lecture prets


## 5. Essai : 3 bulletins par modèle

À faire **avant** toute lecture complète, pour ne pas épuiser les crédits. Un modèle qui échoue dès le premier appel
est abandonné aussitôt. Vérifier pour chaque modèle :

- **`json_ok`** = 1 : il respecte le format de réponse (sinon ses lectures seraient comptées vides) ;
- **`version_servie`** : c'est bien le modèle phare demandé ;
- **`s_par_appel`, `jetons`, `cout`** : multiplier par le nombre de bulletins pour estimer durée et facture.

Les « justes » sur 3 bulletins ne mesurent rien : c'est seulement un signe de bon fonctionnement.

In [5]:
# 5. Essai sur N_ESSAI bulletins par modele
cles_essai = list(APP.cle[:N_ESSAI])
lignes = []
for modele in MODELES_EDEN:
    for cle in cles_essai:
        res, ok, infos = appel(modele, bloc(cle), eden=True, tentatives=1)
        if not ok:
            print(f"{modele} : ECHEC des le premier appel - {infos.get('erreur', '')[:200]}")
            print("  -> modele abandonne (identifiant a verifier dans le catalogue Eden AI)")
            break
        lignes.append({"modele": modele, "version_servie": VERSIONS.get(modele, "?"),
                       "secondes": infos["secondes"], "json_ok": infos["json_ok"],
                       "jetons": infos["jetons"], "cout": infos["cout"],
                       "justes": sum(normaliser(res[c]) == REF[(cle, c)] for c in COLS)})
ESSAI = pd.DataFrame(lignes)
if len(ESSAI):
    resume = ESSAI.groupby(["modele", "version_servie"]).agg(
        appels=("secondes", "size"), s_par_appel=("secondes", "mean"), json_ok=("json_ok", "mean"),
        jetons_par_appel=("jetons", "mean"), justes=("justes", "sum")).round(2)
    display(resume)
    n_total = len(APP)
    for m_, g_ in ESSAI.groupby("modele"):
        couts = pd.to_numeric(g_.cout, errors="coerce").dropna()
        estimation = f"{couts.mean() * n_total:.2f}" if len(couts) else "inconnu (voir le tableau de bord Eden AI)"
        print(f"{m_} : ~{g_.secondes.mean() * n_total / N_PARALLELE / 60:.0f} min et cout estime {estimation} "
              f"pour {n_total} bulletins")
print("\nReporter dans MODELES_RETENUS (cellule 1) les modeles a lancer sur tous les bulletins.")

google/gemini-flash-latest : ECHEC des le premier appel - HTTPError 401 {"detail":"Invalid token"}
  -> modele abandonne (identifiant a verifier dans le catalogue Eden AI)

Reporter dans MODELES_RETENUS (cellule 1) les modeles a lancer sur tous les bulletins.


In [7]:
print("debut de la cle :", os.environ["EDENAI_API_KEY"][:3])

debut de la cle : hvs


In [6]:
k = os.environ.get("EDENAI_API_KEY", "")
print(f"longueur : {len(k)} | espaces en bord : {k != k.strip()} | origine Vault : "
      f"{bool(lire_vault('onyxia-kv/farisazibert/EDENAI'))}")

longueur : 95 | espaces en bord : False | origine Vault : False


In [ ]:
import requests
jeton = os.environ["EDENAI_API_KEY"]
r = requests.post(f"{os.environ['VAULT_ADDR'].rstrip('/')}/v1/auth/token/revoke-self",
                  headers={"X-Vault-Token": jeton}, timeout=30)
print("jeton revoque" if r.status_code == 204 else f"revocation : HTTP {r.status_code}")
os.environ.pop("EDENAI_API_KEY", None)

## 6. gemma4, la référence

Lu par le service du SSP Cloud, avec la même consigne et les mêmes blocs. Gratuit, mais le service peut être
indisponible : si la progression affiche « aucune réponse encore » plusieurs fois, gemma4 est en panne ;
interrompre et relancer plus tard (le cache garde ce qui est lu).

In [ ]:
# 6. gemma4 sur tous les bulletins
LECT = {MODELE_GEMMA: lectures(MODELE_GEMMA, eden=False)}
print(f"{MODELE_GEMMA} : {LECT[MODELE_GEMMA].cle.nunique()}/{len(APP)} bulletins lus")

## 7. Les modèles retenus, sur tous les bulletins

Seuls les modèles de `MODELES_RETENUS` (cellule 1) sont lancés. Conseil : un modèle à la fois, d'abord en
`TEST_RAPIDE = True` (10 bulletins), puis en complet. Le cache évite de payer deux fois le même bulletin.

In [ ]:
# 7. Lectures completes des modeles retenus
if not MODELES_RETENUS:
    print("MODELES_RETENUS est vide : le remplir dans la cellule 1 apres l'essai (etape 5).")
for modele in MODELES_RETENUS:
    LECT[modele] = lectures(modele, eden=True)
print({m_: f"{d.cle.nunique()}/{len(APP)} bulletins" for m_, d in LECT.items()})

## 8. Comparaison

- **justes** : lecture identique au Fiqual ;
- **avoués** : le modèle répond ILLISIBLE (le doute est signalé, un humain peut reprendre) ;
- **erreurs silencieuses** : lecture fausse, présentée comme sûre : l'erreur la plus coûteuse ;
- **vs gemma4** : lignes justes pour ce modèle seul (+) et pour gemma4 seul (−), sur les mêmes lignes ;
  **p** < 0,05 : écart réel avec gemma4 (test de McNemar).

In [ ]:
# 8. Comparaison avec gemma4
def mesurer(L_):
    d = L_.assign(lu_n=L_.lu.map(normaliser))
    d["attendu"] = [REF.get((k_, c), "") for k_, c in zip(d.cle, d.champ)]
    d = d[d.attendu != ""]
    d["juste"] = d.lu_n == d.attendu
    d["avoue"] = d.lu_n == "ILLISIBLE"
    return d.set_index(["cle", "champ"])


ref_g = mesurer(LECT[MODELE_GEMMA])
lignes = []
for modele, L_ in LECT.items():
    d = mesurer(L_)
    ligne = {"modele": modele, "version": VERSIONS.get(modele, modele), "lignes": len(d),
             "justes_%": round(100 * d.juste.mean(), 1), "IC95": ic(int(d.juste.sum()), len(d)),
             "avoues_%": round(100 * d.avoue.mean(), 1),
             "silencieuses_%": round(100 * (~d.juste & ~d.avoue).mean(), 1)}
    if modele != MODELE_GEMMA:
        communs = d.index.intersection(ref_g.index)
        n10, n01, p = mcnemar(d.juste[communs], ref_g.juste[communs])
        ligne |= {"vs gemma4": f"+{n10}/-{n01}", "p": round(p, 4), "lignes_communes": len(communs)}
    lignes.append(ligne)
COMPARAISON = pd.DataFrame(lignes).sort_values("justes_%", ascending=False).reset_index(drop=True)
COMPARAISON.to_csv(RESULTATS / f"comparaison_{CAMPAGNE}{SUFFIXE}.csv", index=False)
with pd.option_context("display.max_columns", 20):
    display(COMPARAISON)
print("\nLire d'abord les erreurs silencieuses, puis la valeur p : sans p < 0,05, l'ecart avec gemma4 "
      "n'est pas etabli.")

## 9. Enregistrement MLflow (projet + personnel)

Seul le tableau de comparaison (des taux, aucune lecture) est envoyé. Les lectures restent dans
`~/work/banc-htr/resultats/eden/`.

In [ ]:
# 9. MLflow : serveur du projet (Vault) et serveur personnel (celui du service)
VARS_MLFLOW = ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
               "MLFLOW_TRACKING_TOKEN", "MLFLOW_TRACKING_INSECURE_TLS")


def config_personnelle():
    try:
        brut_env = open("/proc/1/environ", "rb").read().split(b"\0")
        env = dict(x.decode(errors="ignore").split("=", 1) for x in brut_env if b"=" in x)
    except OSError:
        env = dict(os.environ)
    return {k: env[k] for k in VARS_MLFLOW if k in env}


def config_projet():
    d = lire_vault("onyxia-kv/projet-mesure-qualite-rp/MLFLOW_SECRET")
    return {k: str(v) for k, v in d.items() if k in VARS_MLFLOW}


@contextlib.contextmanager
def sur_serveur(config):
    avant, avant_uri = {k: os.environ.get(k) for k in VARS_MLFLOW}, mlflow.get_tracking_uri()
    try:
        for k in VARS_MLFLOW:
            if k in config:
                os.environ[k] = config[k]
            else:
                os.environ.pop(k, None)
        mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
        yield
    finally:
        for k, v in avant.items():
            os.environ.pop(k, None) if v is None else os.environ.__setitem__(k, v)
        mlflow.set_tracking_uri(avant_uri)


f_comp = RESULTATS / f"comparaison_{CAMPAGNE}{SUFFIXE}.csv"


def enregistrer():
    with mlflow.start_run(run_name=f"grands-modeles-eden-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "grands-modeles", "notebook": "eden", "statut": "exploratoire",
                         "campagne": CAMPAGNE, "donnees": "bulletins fictifs"})
        mlflow.log_params({"bulletins": len(APP), "modeles": ",".join(LECT),
                           "versions": json.dumps(VERSIONS)[:500]})
        for _, r in COMPARAISON.iterrows():
            nom = nom_fichier(r["modele"])[:60]
            for col in ["justes_%", "avoues_%", "silencieuses_%", "p"]:
                v = pd.to_numeric(r.get(col), errors="coerce")
                if pd.notna(v):
                    mlflow.log_metric(f"{nom}_{col.replace('%', 'pct')}", float(v))
        mlflow.log_text(PROMPT_BLOC, "consigne.txt")
        mlflow.log_artifact(str(f_comp))
    return run.info.run_id[:8]


if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre dans MLflow.")
else:
    for nom_serveur, config in [("projet", config_projet()), ("personnel", config_personnelle())]:
        if "MLFLOW_TRACKING_URI" not in config:
            print(f"MLflow {nom_serveur} : non configure, essai non enregistre")
            continue
        try:
            with sur_serveur(config):
                mlflow.set_experiment("fiqual-banc-essai")
                print(f"MLflow {nom_serveur} :", enregistrer())
        except Exception as e:
            print(f"MLflow {nom_serveur} : echec - {type(e).__name__} : {str(e)[:150]}")

## À retenir

- **Un modèle plus gros n'est utile que s'il fait mieux que gemma4 de façon significative** (p < 0,05) **et**
  sans augmenter les erreurs silencieuses.
- **Rapporter le coût et le temps** à côté de la qualité : « X % contre Y % pour gemma4, pour Z € les 1 000 bulletins ».
- **Bulletins fictifs** : le classement des modèles est instructif, mais les taux absolus peuvent différer d'une
  vraie campagne (noms moins variés, écriture plus soignée).
- **Versions** : la colonne `version` dit quel modèle a réellement répondu ; les alias « latest » changent sans prévenir.